## Setup

In [1]:
%load_ext autoreload
%autoreload 2

from pyo.lib.server import Server

from pysynth.clock import Clock
from pysynth.tempo import Tempo
from pysynth.patches import PatchRack, atmosphere, bass, clock_tick, drone, hat, low_hat

s = Server().boot()
s.start()

tempo = Tempo(bpm=142)
rack = PatchRack()

# single shared pulse every patch (except clock_tick, which is deliberately
# free-running) divides down from, so they lock to a common downbeat instead
# of just a common tempo - see src/pysynth/clock.py
clock = Clock(tempo)
clock.start()

Pyo warning: Portaudio input device `MacBook Pro Microphone` has fewer channels (1) than requested (2).
Pyo warning: Portmidi warning: no midi device found!
Portmidi closed.


## Bass

Rolling 16-step bassline through a resonant, LFO-swept lowpass filter. See `src/pysynth/patches/bass.py`.

In [7]:
bass_patch = rack.toggle(
    "bass",
    bass.build,
    tempo,
    clock,
    root_freq=bass.ROOT_FREQ,
    filter_res=0.75,
    filter_base=900,
    filter_range=400,
)

## Atmosphere (FM pad + arpeggiator)

FM pad voice arpeggiated at 8th notes, with a slow amplitude swell and reverb. See `src/pysynth/patches/atmosphere.py`.

In [44]:
atmosphere_patch = rack.toggle(
    "atmosphere",
    atmosphere.build,
    tempo,
    clock,
    step_division=clock.FOURTH,
    arp_root=atmosphere.ARP_ROOT,
    fm_ratio=0.5012,
    fm_index=4,
    reverb_size=0.9,
    reverb_damp=0.9,
    reverb_bal=0.9,
)

## Hi-hat

Subtle high-passed noise tick, once per 8th note, for top-end texture. See `src/pysynth/patches/hat.py`.

In [45]:
hat_patch = rack.toggle(
    "hat",
    hat.build,
    tempo,
    clock,
    cutoff_freq=hat.CUTOFF_FREQ,
    level=0.2,
    decay=0.2,
)

## Low hat

Darker noise tick, once per quarter note, as a rarer, dubbier accent. See `src/pysynth/patches/low_hat.py`.

In [ ]:
low_hat_patch = rack.toggle(
    "low_hat",
    low_hat.build,
    tempo,
    clock,
    cutoff_freq=low_hat.CUTOFF_FREQ,
    level=0.3,
    decay=0.5,
)

## Clock tick

Pink Floyd "Time"-style clock shop: four noise ticks, each on its own free-running period in real seconds rather than the tempo grid, so they drift past each other and never share a downbeat — a quiet background texture, not a groove element. See `src/pysynth/patches/clock_tick.py`.

In [ ]:
clock_tick_patch = rack.toggle(
    "clock_tick",
    clock_tick.build,
    tempo,
    level=clock_tick.OVERALL_LEVEL,
    wood_q=3,
    glass_q=6,
)

## Drone

Slow-winding FM drone: note changes once every 8 bars, with a continuously drifting timbre. See `src/pysynth/patches/drone.py`.

In [ ]:
drone_patch = rack.toggle(
    "drone",
    drone.build,
    tempo,
    clock,
    root_freq=drone.DRONE_ROOT,
    reverb_size=0.9,
    reverb_damp=0.3,
    reverb_bal=0.9,
)

## Stop

Each patch can be stopped/restarted on its own by rerunning `rack.stop("<name>")` / re-running its build cell. This cell stops everything at once.

In [ ]:
agr

In [ ]:
rack.stop_all()

<!-- ## Panic reset

If a sound is stuck playing and `rack.stop_all()` doesn't touch it — most likely it was started before this notebook adopted `PatchRack`, so nothing currently holds a reference to it — the fix is to restart the audio server outright. This tears down the entire audio graph regardless of what Python still references, so you'll need to rerun every patch cell afterwards (they'll pick up `rack` again since that variable survives). -->

In [ ]:
# s.stop()
# s.shutdown()
# s.boot()
# s.start()

In [ ]:
# s.stop()